# Tải **Training Set A** và **Training Set B**

In [1]:

DOWNLOAD_SET_A = True
DOWNLOAD_SET_B = True

OUT_DIR_A      = "/kaggle/working/Data-Set-A"
OUT_DIR_B      = "/kaggle/working/Data-Set-B"

MAX_WORKERS    = 8      
TIMEOUT        = 30     
MAX_RETRIES    = 5
BACKOFF_FACTOR = 1.0
THROTTLE       = 0.0    
SKIP_IF_EXISTS = True   

BASE_URL_A     = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA"
BASE_URL_B     = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setB"

FNAME_TMPL_A   = "p{idx:06d}.psv"   # p000001.psv → p020643.psv
FNAME_TMPL_B   = "p1{idx:05d}.psv"  # p100001.psv → p120000.psv

SET_A_START, SET_A_END = 1, 20643
SET_B_START, SET_B_END = 1, 20000



In [2]:
import os, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.notebook import tqdm


def build_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=BACKOFF_FACTOR,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "HEAD"]),
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://",  HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": "physionet-downloader/1.0"})
    return s


def download_one(session, url: str, out_path: Path) -> str:
    """Tải 1 file. Trả về: ok | skipped | not_found | error:<msg>"""
    if SKIP_IF_EXISTS and out_path.exists() and out_path.stat().st_size > 0:
        return "skipped"
    tmp = out_path.with_suffix(".part")
    try:
        with session.get(url, stream=True, timeout=TIMEOUT) as r:
            if r.status_code == 200:
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=16_384):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, out_path)
                if THROTTLE:
                    time.sleep(THROTTLE)
                return "ok"
            elif r.status_code == 404:
                return "not_found"
            else:
                return f"http_{r.status_code}"
    except Exception as e:
        return f"error:{e}"
    finally:
        if tmp.exists():
            try: tmp.unlink()
            except: pass


def bulk_download(base_url, fname_tmpl, start, end, out_dir, label):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    tasks = [
        (f"{base_url}/{fname_tmpl.format(idx=i)}", out_dir / fname_tmpl.format(idx=i))
        for i in range(start, end + 1)
    ]

    results = {}
    session = build_session()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(download_one, session, url, path): (url, path) for url, path in tasks}
        with tqdm(total=len(tasks), desc=label, unit="file") as pbar:
            for fut in as_completed(futures):
                url, path = futures[fut]
                try:
                    status = fut.result()
                except Exception as e:
                    status = f"exception:{e}"
                results[path.name] = status
                pbar.update(1)

    # Tổng kết
    ok       = sum(1 for v in results.values() if v == "ok")
    skipped  = sum(1 for v in results.values() if v == "skipped")
    notfound = sum(1 for v in results.values() if v == "not_found")
    errors   = [(k, v) for k, v in results.items() if v not in ("ok", "skipped", "not_found")]

    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"   OK        : {ok:>7,}")
    print(f"    Skipped  : {skipped:>7,}")
    print(f"   Not found : {notfound:>7,}")
    print(f"   Lỗi       : {len(errors):>7,}")
    if errors:
        print(f"\n  Mẫu lỗi (10 đầu):")
        for name, status in errors[:10]:
            print(f"    {name} → {status}")
    print(f"{'='*40}")


print('Sẵn sàng tải.')

Sẵn sàng tải.


In [3]:
# Tải Training Set A
if DOWNLOAD_SET_A:
    bulk_download(BASE_URL_A, FNAME_TMPL_A, SET_A_START, SET_A_END, OUT_DIR_A, "Training Set A")

Training Set A:   0%|          | 0/20643 [00:00<?, ?file/s]


  Training Set A
   OK        :  20,336
    Skipped  :       0
   Not found :     307
   Lỗi       :       0


In [4]:
# Tải Training Set B
if DOWNLOAD_SET_B:
    bulk_download(BASE_URL_B, FNAME_TMPL_B, SET_B_START, SET_B_END, OUT_DIR_B, "Training Set B")

Training Set B:   0%|          | 0/20000 [00:00<?, ?file/s]


  Training Set B
   OK        :  20,000
    Skipped  :       0
   Not found :       0
   Lỗi       :       0


In [5]:
# Kiểm tra nhanh dữ liệu
import pandas as pd

for label, directory in [("Set A", OUT_DIR_A), ("Set B", OUT_DIR_B)]:
    files = sorted(Path(directory).glob("*.psv"))
    print(f"\n {label}: {len(files):,} files")
    if files:
        df = pd.read_csv(files[0], sep="|")
        print(f"   File mẫu: {files[0].name} — {df.shape[0]} dòng, {df.shape[1]} cột")
        print(f"   Cột: {list(df.columns)}")


 Set A: 20,336 files
   File mẫu: p000001.psv — 54 dòng, 41 cột
   Cột: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']

 Set B: 20,000 files
   File mẫu: p100001.psv — 24 dòng, 41 cột
   Cột: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']
